In [4]:
# Let's read the raw data
import os
BASE_DIR = os.getcwd()
PARENT_DIR = os.path.dirname(BASE_DIR)
DATA_DIR = os.path.join(PARENT_DIR, 'data', 'data_cleaning')
DATA_DIR

'd:\\python\\AI\\pynb\\data\\data_cleaning'

In [6]:
csv_path = os.path.join(DATA_DIR, 'BL-Flickr-Images-Book.csv')
csv_path

'd:\\python\\AI\\pynb\\data\\data_cleaning\\BL-Flickr-Images-Book.csv'

In [7]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent 

d:\python\AI\pynb\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\python\AI\pynb\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [28]:
df = pd.read_csv(csv_path)
df.head()

,Identifier,Edition Statement,Place of Publication,Date of Publication,Publisher,Title,Author,Contributors,Corporate Author,Corporate Contributors,Former owner,Engraver,Issuance type,Flickr URL,Shelfmarks
0,206,NaN,London,1879 [1878],S. Tinsley & Co.,Walter Forbes. [A novel.] By A. A,A. A.,"FORBES, Walter.",NaN,NaN,NaN,NaN,monographic,http://www.flickr.com/photos/britishlibrary/ta...,British Library HMNTS 12641.b.30.
1,216,NaN,London; Virtue & Yorston,1868,Virtue & Co.,All for Greed. [A novel. The dedication signed...,"A., A. A.","BLAZE DE BURY, Marie Pauline Rose - Baroness",NaN,NaN,NaN,NaN,monographic,http://www.flickr.com/photos/britishlibrary/ta...,British Library HMNTS 12626.cc.2.
2,218,NaN,London,1869,"Bradbury, Evans & Co.",Love the Avenger. By the author of “All for Gr...,"A., A. A.","BLAZE DE BURY, Marie Pauline Rose - Baroness",NaN,NaN,NaN,NaN,monographic,http://www.flickr.com/photos/britishlibrary/ta...,British Library HMNTS 12625.dd.1.
3,472,NaN,London,1851,James Darling,"Welsh Sketches, chiefly ecclesiastical, to the...","A., E. S.","Appleyard, Ernest Silvanus.",NaN,NaN,NaN,NaN,monographic,http://www.flickr.com/photos/britishlibrary/ta...,British Library HMNTS 10369.bbb.15.
4,480,"A new edition, revised, etc.",London,1857,Wertheim & Macintosh,"[The World in which I live, and my place in it...","A., E. S.","BROOME, John Henry.",NaN,NaN,NaN,NaN,monographic,http://www.flickr.com/photos/britishlibrary/ta...,British Library HMNTS 9007.d.28.


In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.3-70b-versatile")


In [19]:
prefix = """You are a Data Profiling Agent.

Your task is to analyze a Pandas DataFrame dynamically and produce a comprehensive data-quality profile.

You will receive:

1. The DataFrame schema
2. Column data types
3. Number of rows and columns
4. Missing-value statistics
5. Unique-value statistics
6. Sample values from each column
7. Basic statistical information
8. Duplicate-row information

Do NOT assume any fixed column names or schema. The DataFrame can represent any type of business data.

For each column, analyze:

* Inferred semantic meaning (e.g., name, email, age, date, currency, ID, category)
* Actual Pandas data type
* Likely expected data type
* Missing-value percentage
* Duplicate-value percentage
* Number of unique values
* Whether the column appears to be an identifier
* Whether the column contains inconsistent formatting
* Whether values contain leading/trailing whitespace
* Whether categorical values have inconsistent representations
  (e.g., "Male", "male", "M")
* Whether numeric values appear to be stored as strings
* Whether date values appear to be stored as strings
* Whether there are suspicious or invalid values
* Whether there are potential outliers
* Whether the column has high cardinality
* Whether the column appears unnecessary or redundant

Also analyze the DataFrame as a whole:

* Duplicate rows
* Completely empty rows
* Completely empty columns
* Columns with excessive missing values
* Potentially redundant columns
* Potential relationships between columns
* Potential data-quality problems
* Potential schema inconsistencies

For every detected issue, provide:

* issue_type
* column
* severity
* description
* evidence
* recommended_action

Severity must be one of:

* critical
* high
* medium
* low
* info

IMPORTANT RULES:

1. Do not modify the DataFrame.
2. Do not invent information that is not supported by the profiling data.
3. Clearly distinguish between confirmed issues and potential issues.
4. Do not assume that an outlier is necessarily an error.
5. Do not automatically recommend deleting data unless there is strong evidence.
6. Consider the semantic meaning of the column before recommending a cleaning operation.
7. Return your result in the specified structured format.

"""

suffix = '''Return the following JSON structure:

{{
"dataset_summary": {{
"rows": 0,
"columns": 0,
"duplicate_rows": 0,
"overall_quality": "good|fair|poor|critical"
}},

```
"columns": [
    {{
        "name": "",
        "actual_dtype": "",
        "inferred_type": "",
        "semantic_role": "",
        "missing_percentage": 0,
        "unique_count": 0,
        "high_cardinality": false,
        "potential_issues": []
    }}
],

"issues": [
    {{
        "issue_type": "",
        "column": "",
        "severity": "",
        "description": "",
        "evidence": "",
        "recommended_action": ""
    }}
],

"cleaning_recommendations": [
    {{
        "column": "",
        "operation": "",
        "reason": "",
        "confidence": 0.0
    }}
]
```

}}
'''

In [29]:
agent = create_pandas_dataframe_agent(
    llm,
    df,
    agent_type="tool-calling",
    allow_dangerous_code=True,
    verbose=True,
)

In [32]:
resp = agent.invoke("Analyse the entire dateset and profile it for cleaning")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'import pandas as pd\nimport numpy as np\n\ndf = pd.DataFrame({"Identifier": [206, 216, 218, 472, 480], "Edition Statement": [np.nan, np.nan, np.nan, np.nan, "A new edition, revised, etc."], "Place of Publication": ["London", "London; Virtue & Yorston", "London", "London", "London"], "Date of Publication": ["1879 [1878]", "1868", "1869", "1851", "1857"], "Publisher": ["S. Tinsley & Co.", "Virtue & Co.", "Bradbury, Evans & Co.", "James Darling", "Wertheim & Macintosh"], "Title": ["Walter Forbes. [A novel.] By A. A.", "All for Greed. [A novel. The dedication signed: A. A. A., i.e. Marie Pauline Rose, Baroness Blaze de Bury.]", "Love the Avenger. By the author of “All for Greed.” [The dedication signed: A. A. A., i.e. Marie Pauline Rose, Baroness Blaze de Bury.]", "Welsh Sketches, chiefly ecclesiastical, to the close of the twelfth century. By the author of “Proposals for Christian Union” (E. S. A. [i.e. 